In [10]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.metrics.pairwise import cosine_similarity
import operator
%matplotlib inline

In [11]:
data=pd.read_csv('dataset/train.csv',header=None)

In [12]:
data.columns=['user','song','frequency']

###### For collaborative filtering we'll need to create a pivot table of users on one axis and songs along the other. The pivot table will help us in defining the similarity between users and songs to better predict who will like what.

In [13]:
piv = data.pivot_table(index=['user'], columns=['song'], values='frequency')

In [14]:
print(piv.shape)
piv.head()

(937, 15580)


song,0,1,2,3,4,5,6,7,8,9,...,15658,15659,15660,15661,15662,15663,15664,15665,15666,15667
user,,,,,,,,,,,,,,,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


###### We have used normalisation to normalize the pivot table

In [19]:
piv_norm = piv.apply(lambda x: (x-np.mean(x))/(np.max(x)-np.min(x)), axis=1)

piv_norm.fillna(0, inplace=True)
piv_norm = piv_norm.T
piv_norm = piv_norm.loc[:, (piv_norm != 0).any(axis=0)]

###### we are creating a sparse matrix from normalised  pivot table.

In [20]:
piv_sparse = sp.sparse.csr_matrix(piv_norm.values)

###### These matrices show us the computed cosine similarity values between each user/user array pair and item/item array pair.

In [21]:
song_similarity = cosine_similarity(piv_sparse)
user_similarity = cosine_similarity(piv_sparse.T)

##### we are transforming the song and user similarity from matrices to dataframes

In [22]:
song_sim_df = pd.DataFrame(song_similarity, index = piv_norm.index, columns = piv_norm.index)
user_sim_df = pd.DataFrame(user_similarity, index = piv_norm.columns, columns = piv_norm.columns)

In [40]:
song_sim_df.head()

song,0,1,2,3,4,5,6,7,8,9,...,15658,15659,15660,15661,15662,15663,15664,15665,15666,15667
song,,,,,,,,,,,,,,,,,,,,,
0,1.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0
1,0.0,1.000000,0.0,-0.005072,-0.001793,-0.049835,0.032627,0.0,-0.038623,0.009727,...,-0.002723,-0.010838,0.001740,0.000000,0.0,0.000000,-0.000563,0.0,0.014514,0.0
2,0.0,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0
3,0.0,-0.005072,0.0,1.000000,0.001323,0.000000,-0.242559,0.0,0.039177,-0.068832,...,-0.020552,0.000000,0.002339,0.014741,0.0,0.006516,0.000000,0.0,0.000000,0.0
4,0.0,-0.001793,0.0,0.001323,1.000000,0.000000,0.000000,0.0,0.000000,0.016829,...,0.058204,0.000000,0.000000,0.277567,0.0,0.000000,0.000000,0.0,-0.012524,0.0


In [41]:
user_sim_df.head()

user,0,1,2,3,4,5,6,7,8,9,...,982,983,984,985,986,987,988,989,990,991
user,,,,,,,,,,,,,,,,,,,,,
0,1.000000,-0.000830,0.001023,0.103987,0.000284,0.007177,0.000810,0.003961,0.002264,0.000338,...,-0.002935,-0.005211,0.084891,-0.010768,0.000985,0.114062,-0.002566,0.080903,-0.000003,0.033787
1,-0.000830,1.000000,0.032660,-0.008364,0.007746,0.041975,-0.003679,-0.021570,-0.005560,0.001181,...,-0.013711,0.003673,0.060637,0.050631,0.036037,-0.004256,-0.003268,0.167363,0.005215,0.065715
2,0.001023,0.032660,1.000000,0.036612,0.088180,-0.000735,-0.005086,-0.004426,0.005116,-0.001042,...,0.008483,0.029076,-0.005219,0.028617,0.004805,0.061116,-0.003836,0.044566,0.003152,0.037678
3,0.103987,-0.008364,0.036612,1.000000,0.026146,0.023934,0.001732,-0.009482,-0.009773,0.000746,...,-0.011380,0.099785,0.130350,-0.003522,0.053797,0.410572,-0.002362,0.287761,-0.018081,0.121299
4,0.000284,0.007746,0.088180,0.026146,1.000000,-0.026352,-0.004179,-0.057272,-0.010783,-0.009225,...,-0.003827,-0.013957,0.109217,0.006914,0.010051,-0.002996,-0.001126,-0.012225,-0.014696,-0.011316


##### This function will return the top 10 shows with the highest cosine similarity value

In [23]:
def top_songs(song):
    count = 1
    print('Similar songs to {} include:\n'.format(song))
    for item in song_sim_df.sort_values(by = song, ascending = False).index[1:11]:
        print('No. {}: {}'.format(count, item))
        count +=1 

 ##### This function will return the top users with the highest similarity value 

In [24]:
def top_users(user):
    
    if user not in piv_norm.columns:
        return('No data available on user {}'.format(user))
    
    print('Most Similar Users:\n')
    sim_values = user_sim_df.sort_values(by=user, ascending=False).loc[:,user].tolist()[1:11]
    sim_users = user_sim_df.sort_values(by=user, ascending=False).index[1:11]
    zipped = zip(sim_users, sim_values,)
    for suser, sim in zipped:
        print('User #{0}, Similarity value: {1:.2f}'.format(suser, sim)) 

##### This function constructs a list of lists containing the highest played songs per similar user and returns the name of the show along with the frequency it appears in the list

In [25]:
def similar_user_recs(user):
    
    if user not in piv_norm.columns:
        return('No data available on user {}'.format(user))
    
    sim_users = user_sim_df.sort_values(by=user, ascending=False).index[1:11]
    best = []
    most_common = {}
    
    for i in sim_users:
        max_score = piv_norm.loc[:, i].max()
        best.append(piv_norm[piv_norm.loc[:, i]==max_score].index.tolist())
    for i in range(len(best)):
        for j in best[i]:
            if j in most_common:
                most_common[j] += 1
            else:
                most_common[j] = 1
    sorted_list = sorted(most_common.items(), key=operator.itemgetter(1), reverse=True)
    return sorted_list[:5]    

##### This function calculates the weighted average of similar users to determine a potential rating for an input user and show

In [26]:
def predicted_frequency(song, user):
    sim_users = user_sim_df.sort_values(by=user, ascending=False).index[1:1000]
    user_values = user_sim_df.sort_values(by=user, ascending=False).loc[:,user].tolist()[1:1000]
    freq_list = []
    weight_list = []
    for j, i in enumerate(sim_users):
        freq = piv.loc[i, song]
        similarity = user_values[j]
        if np.isnan(freq):
            continue
        elif not np.isnan(freq):
            freq_list.append(freq*similarity)
            weight_list.append(similarity)
    return sum(freq_list)/sum(weight_list) 

In [27]:
top_songs(1)

Similar songs to 1 include:

No. 1: 4096
No. 2: 14247
No. 3: 12408
No. 4: 4954
No. 5: 6342
No. 6: 5334
No. 7: 12904
No. 8: 11916
No. 9: 8663
No. 10: 5327


In [28]:
top_users(19)

Most Similar Users:

User #617, Similarity value: 0.74
User #675, Similarity value: 0.70
User #651, Similarity value: 0.69
User #138, Similarity value: 0.63
User #324, Similarity value: 0.61
User #885, Similarity value: 0.54
User #699, Similarity value: 0.48
User #536, Similarity value: 0.46
User #92, Similarity value: 0.42
User #347, Similarity value: 0.39


In [29]:
similar_user_recs(19)

[(9123, 8), (8237, 1), (10548, 1)]

In [30]:
predicted_frequency(3388, 19)

134.61458439554627

In [31]:
predicted_frequency(19, 19)

151.0

In [32]:
predicted_frequency(0, 19)

48.50472182512909

In [33]:
predicted_frequency(1, 19)

18.589085237348456

In [36]:
listened = piv.T[piv.loc[5,:]>0].index.tolist()
errors = []
for i in listened:
    actual=piv.loc[5, i]
    predicted = predicted_frequency(i,5)
    errors.append((actual-predicted))
sum(errors)/len(listened)

25.93337580920784

In [38]:
listened = piv.T[piv.loc[3,:]>0].index.tolist()
errors = []
for i in listened:
    actual=piv.loc[3, i]
    predicted = predicted_frequency(i,3)
    errors.append((actual-predicted))
sum(errors)/len(listened)

-13.352941268483107